[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-11b-grpo-on-a-toy-policy.ipynb)

# Advanced Discussion 11 (continued) — GRPO with a checker

Group-relative advantages, empty groups, group size, the standard-deviation term, clipping and the KL penalty on a tabular policy.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Pure PyTorch; the sweeps take about a minute.

In [ ]:
%pip install -q torch numpy

In [ ]:
import numpy as np, torch, torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0)

## 1. the setting: verifiable rewards. 6 prompts, 6 candidate answers each, exactly one is correct (a checker says so)

In [ ]:
P, A = 6, 6
correct = torch.tensor([0, 1, 2, 3, 4, 5])                               # index of the correct answer for each prompt
p_ref = torch.tensor([0.02, 0.05, 0.15, 0.30, 0.50, 0.70])               # how often the starting (SFT) policy is right, per prompt
def make_ref_logits():
    L = torch.zeros(P, A)
    for i in range(P):
        rest = (1 - p_ref[i]) / (A - 1); probs = torch.full((A,), rest.item()); probs[correct[i]] = p_ref[i]; L[i] = probs.log()
    return L
REF = make_ref_logits()
print("starting pass rate per prompt:", p_ref.tolist(), " mean %.3f" % p_ref.mean())

## 2. one group, worked by hand: 8 samples for one prompt, 1 of them correct

In [ ]:
r = torch.tensor([1., 0, 0, 0, 0, 0, 0, 0])
mean, std = r.mean(), r.std(unbiased=False)
print("\nworked group (8 samples, 1 correct): mean %.3f, std %.3f" % (mean, std))
print("  GRPO advantages (r - mean) / std : correct %+.3f, each wrong %+.3f" % ((1 - mean) / std, (0 - mean) / std))
print("  without dividing by std          : correct %+.3f, each wrong %+.3f" % (1 - mean, -mean))
r2 = torch.tensor([1., 1, 1, 1, 0, 0, 0, 0]); m2, s2 = r2.mean(), r2.std(unbiased=False)
print("  a 4-of-8 group, GRPO             : correct %+.3f, each wrong %+.3f" % ((1 - m2) / s2, (0 - m2) / s2))
print("  an all-wrong or all-right group  : every advantage is 0 -> no learning signal")

## 3. how often does a group carry no signal? p^G + (1-p)^G

In [ ]:
print("\nP(group has no signal) = p^G + (1-p)^G")
print("  p      G=2     G=4     G=8     G=16    G=32")
for p in (0.02, 0.05, 0.15, 0.30, 0.50, 0.70, 0.95):
    print("  %.2f  " % p + "  ".join("%6.3f" % (p ** G + (1 - p) ** G) for G in (2, 4, 8, 16, 32)))

## 4. train: GRPO with different group sizes, vs REINFORCE with a single global baseline

In [ ]:
def train(method, G=8, steps=300, lr=0.4, clip=0.2, beta=0.0, std_norm=True, seed=0, inner=1):
    g = torch.Generator().manual_seed(seed); theta = REF.clone().requires_grad_(True); opt = torch.optim.SGD([theta], lr=lr); curve = []; nosig = 0; tot = 0
    for t in range(steps):
        pi_old = F.softmax(theta.detach(), 1)
        a = torch.multinomial(pi_old, G, replacement=True, generator=g)                       # (P, G) sampled answers
        rew = (a == correct[:, None]).float()                                                # verifiable 0/1 reward
        if method == "grpo":
            mu = rew.mean(1, keepdim=True); sd = rew.std(1, unbiased=False, keepdim=True)
            adv = (rew - mu) / (sd + 1e-4) if std_norm else (rew - mu)
        else:                                                                               # REINFORCE, one global mean baseline
            adv = rew - rew.mean()
        nosig += ((rew.mean(1) == 0) | (rew.mean(1) == 1)).sum().item(); tot += P
        logp_old = pi_old.log().gather(1, a)
        for _ in range(inner):
            logp = F.log_softmax(theta, 1).gather(1, a); ratio = torch.exp(logp - logp_old)
            obj = torch.min(ratio * adv, ratio.clamp(1 - clip, 1 + clip) * adv) if method == "grpo" else logp * adv
            pi_now = F.softmax(theta, 1); klt = (pi_now * (pi_now.clamp_min(1e-9).log() - F.softmax(REF, 1).log())).sum(1).mean() if beta else 0.0
            loss = -obj.mean() + beta * klt; opt.zero_grad(); loss.backward(); opt.step()
        curve.append(F.softmax(theta.detach(), 1)[torch.arange(P), correct])
    final = F.softmax(theta.detach(), 1)[torch.arange(P), correct]
    kl_ = (F.softmax(theta.detach(), 1) * (F.log_softmax(theta.detach(), 1) - F.log_softmax(REF, 1))).sum(1).mean().item()
    return final, kl_, nosig / tot
def avg(method, seeds=12, **kw):
    fs = []; kls = []; ns = []
    for s in range(seeds):
        f, k, n = train(method, seed=s, **kw); fs.append(f); kls.append(k); ns.append(n)
    return torch.stack(fs).mean(0), float(np.mean(kls)), float(np.mean(ns))
print("\nafter 300 steps (mean of 12 seeds): pass rate per prompt, average, and average KL to the starting policy")
print("method                                  " + "  ".join("p%d" % i for i in range(P)) + "    mean    KL     groups with no signal")
for name, m, kw in (("REINFORCE, global baseline, G=8", "rf", dict(G=8)), ("GRPO G=2", "grpo", dict(G=2)), ("GRPO G=4", "grpo", dict(G=4)), ("GRPO G=8", "grpo", dict(G=8)),
                    ("GRPO G=16", "grpo", dict(G=16)), ("GRPO G=32", "grpo", dict(G=32)), ("GRPO G=8, no std division", "grpo", dict(G=8, std_norm=False))):
    f, k, n = avg(m, **kw); print("%-38s  %s   %.3f  %.2f    %.2f" % (name, " ".join("%.2f" % x for x in f), f.mean(), k, n))

print("is the std division just a bigger step? GRPO G=8 without std division, learning rate raised:")
for lr in (0.4, 1.0, 2.0, 4.0):
    f, k, n = avg("grpo", G=8, std_norm=False, lr=lr); print("  lr %.1f  mean pass rate %.3f   KL %.2f" % (lr, f.mean(), k))

## 5. what each term of the objective does: clipping with several update passes, and a KL penalty

In [ ]:
print("\nGRPO G=8 with 4 update passes per batch (reusing samples), by clip range")
for clip in (None, 0.2, 0.05):
    kw = dict(G=8, inner=4, lr=0.4)
    if clip is None: kw["clip"] = 1e9
    else: kw["clip"] = clip
    f, k, n = avg("grpo", **kw); print("  clip %-5s  mean pass rate %.3f   KL %.2f" % (clip, f.mean(), k))
print("\nGRPO G=8 with a KL penalty toward the starting policy")
for beta in (0.0, 0.05, 0.3, 1.0):
    f, k, n = avg("grpo", G=8, beta=beta); print("  beta %.2f   mean pass rate %.3f   KL %.2f" % (beta, f.mean(), k))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.